# Distill OSRM drive times (surrogate for browser heatmaps)

Notebook `03` materializes **seconds** of driving time from each sampled address to each active pharmacy (`osrm_address_pharmacy_duration_sec.npy`). Here we **subsample** address–pharmacy pairs, build cheap **geographic features** (normalized **address and pharmacy** coordinates, haversine distance, initial bearing as **sin/cos**), and fit a **ridge regression** with **standardized** inputs.

**Goal:** export [`../data/drive_time_surrogate.json`](../data/drive_time_surrogate.json) so a small **linear** forward pass in the browser can approximate OSRM for a **user-selected pharmacy** without calling a routing API.

**Validation:** hold out ~20% of **addresses** (not random cells) so scores reflect generalization to new locations.

In [8]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("../data")
DURATION_NPY = DATA_DIR / "osrm_address_pharmacy_duration_sec.npy"
ADDRESSES_CSV = DATA_DIR / "addresses_with_population_weights.csv"
PHARM_JSON = DATA_DIR / "pharmacies_active.json"
OUT_JSON = DATA_DIR / "drive_time_surrogate.json"

RNG = np.random.default_rng(42)
N_SAMPLE = 250_000
RIDGE_ALPHA = 5.0

In [9]:
def haversine_km(lat1: np.ndarray, lon1: np.ndarray, lat2: np.ndarray, lon2: np.ndarray) -> np.ndarray:
    r = 6371.0
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return (2 * r * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))).astype(np.float64)


def bearing_sin_cos(
    lat1: np.ndarray, lon1: np.ndarray, lat2: np.ndarray, lon2: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """Initial bearing from (lat1, lon1) toward (lat2, lon2); sin/cos of angle clockwise from north."""
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dl = np.radians(lon2 - lon1)
    x = np.sin(dl) * np.cos(p2)
    y = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
    theta = np.arctan2(x, y)
    sin_b, cos_b = np.sin(theta), np.cos(theta)
    degenerate = (np.hypot(x, y) < 1e-15) | ~np.isfinite(theta)
    sin_b = np.where(degenerate, 0.0, sin_b)
    cos_b = np.where(degenerate, 1.0, cos_b)
    return sin_b.astype(np.float64), cos_b.astype(np.float64)

In [10]:
D = np.load(DURATION_NPY)
addrs = pd.read_csv(ADDRESSES_CSV)
with PHARM_JSON.open() as f:
    pharm = json.load(f)

assert D.shape == (len(addrs), len(pharm)), (D.shape, len(addrs), len(pharm))
plat = np.array([p["lat"] for p in pharm], dtype=np.float64)
plon = np.array([p["lon"] for p in pharm], dtype=np.float64)

lat_min = float(addrs["lat"].min())
lat_max = float(addrs["lat"].max())
lon_min = float(addrs["lon"].min())
lon_max = float(addrs["lon"].max())
dlat = lat_max - lat_min
dlon = lon_max - lon_min

print(f"matrix {D.shape[0]:,} x {D.shape[1]:,}; bbox lat [{lat_min:.4f}, {lat_max:.4f}] lon [{lon_min:.4f}, {lon_max:.4f}]")

matrix 86,672 x 224; bbox lat [44.0145, 45.0130] lon [-72.9292, -71.4715]


In [11]:
i = RNG.integers(0, D.shape[0], size=N_SAMPLE)
j = RNG.integers(0, D.shape[1], size=N_SAMPLE)
y = D[i, j]
finite = np.isfinite(y)
i, j, y = i[finite], j[finite], y[finite].astype(np.float64)

lat_a = addrs["lat"].to_numpy(dtype=np.float64)[i]
lon_a = addrs["lon"].to_numpy(dtype=np.float64)[i]
lat_p, lon_p = plat[j], plon[j]

hav = haversine_km(lat_a, lon_a, lat_p, lon_p)
norm_lat = (lat_a - lat_min) / dlat
norm_lon = (lon_a - lon_min) / dlon
norm_pharm_lat = (lat_p - lat_min) / dlat
norm_pharm_lon = (lon_p - lon_min) / dlon
sin_b, cos_b = bearing_sin_cos(lat_a, lon_a, lat_p, lon_p)
X = np.column_stack(
    [norm_lat, norm_lon, norm_pharm_lat, norm_pharm_lon, hav, np.log1p(hav), sin_b, cos_b]
)

feature_order = [
    "norm_lat",
    "norm_lon",
    "norm_pharm_lat",
    "norm_pharm_lon",
    "haversine_km",
    "log1p_haversine_km",
    "sin_bearing",
    "cos_bearing",
]
print(f"finite samples: {len(y):,}; features: {feature_order}")

finite samples: 250,000; features: ['norm_lat', 'norm_lon', 'haversine_km', 'log1p_haversine_km', 'sin_bearing', 'cos_bearing']


In [12]:
uniq_addrs = np.unique(i)
RNG.shuffle(uniq_addrs)
n_test_addrs = max(1, int(0.2 * len(uniq_addrs)))
test_addr_set = set(uniq_addrs[:n_test_addrs].tolist())
train_mask = np.array([int(a) not in test_addr_set for a in i], dtype=bool)

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]
print(f"train rows {len(y_train):,}; test rows {len(y_test):,}; held-out addresses {n_test_addrs:,}")

train rows 200,234; test rows 49,766; held-out addresses 16,343


In [13]:
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=42)),
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = root_mean_squared_error(y_test, pred)
print(f"holdout MAE: {mae:.2f} s  RMSE: {rmse:.2f} s")

holdout MAE: 842.73 s  RMSE: 1098.07 s


In [7]:
scale = pipe.named_steps["scale"]
ridge = pipe.named_steps["ridge"]
coef = np.ravel(ridge.coef_)

payload = {
    "schema_version": 3,
    "model": "ridge_standard_scaled",
    "target": "duration_sec_osrm_table",
    "bbox": {
        "lat_min": lat_min,
        "lat_max": lat_max,
        "lon_min": lon_min,
        "lon_max": lon_max,
    },
    "feature_order": feature_order,
    "scaler_mean": scale.mean_.astype(float).tolist(),
    "scaler_scale": scale.scale_.astype(float).tolist(),
    "coef": coef.astype(float).tolist(),
    "intercept": float(ridge.intercept_),
    "training_notes": {
        "n_train": int(train_mask.sum()),
        "n_test": int((~train_mask).sum()),
        "mean_y_sec_holdout": float(np.mean(y_test)),
        "mae_sec_holdout": float(mae),
        "rmse_sec_holdout": float(rmse),
        "ridge_alpha": RIDGE_ALPHA,
        "n_sample_requested": N_SAMPLE,
    },
}

OUT_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"wrote {OUT_JSON}")

wrote ../data/drive_time_surrogate.json


## Browser parity

Use [`../lib/driveTimeSurrogate.ts`](../lib/driveTimeSurrogate.ts): build the same feature vector as `feature_order` (query and **selected pharmacy** normalized with **this** bbox; haversine + `log1p`; **sin/cos** of initial bearing from query → pharmacy), standardize with `scaler_mean` / `scaler_scale`, then dot with `coef` plus `intercept`. Clamp at zero for display.